# Ollama on a Kaggle GPU for Project 5 (agents A1 / A2)

Runs `qwen2.5:3b` and `qwen2.5:7b` on Kaggle's GPU and exposes the Ollama API through a temporary Cloudflare tunnel. Blender, the tools and the checks stay on the laptop; only the model calls come here.

**Before running:** Settings → Accelerator → **GPU T4 x2** (or P100), and Settings → **Internet on** (needs a phone-verified account).

Then *Run all*. Copy the `https://….trycloudflare.com` URL printed by cell 4 and on the laptop run:
```
$env:P5_OLLAMA_URL = "https://….trycloudflare.com"; python eval.py --agent A1
```
Keep the last cell running (it keeps the session alive). The URL is unguessable but public: stop the session when done.

In [ ]:
# 1. GPU check and Ollama install
!nvidia-smi --query-gpu=name,memory.total --format=csv
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 2. Start the Ollama server (models stay loaded for 2 h between calls)
import os, subprocess, time, urllib.request
env = {**os.environ, 'OLLAMA_HOST': '0.0.0.0:11434', 'OLLAMA_KEEP_ALIVE': '2h', 'OLLAMA_ORIGINS': '*'}
server = subprocess.Popen(['ollama', 'serve'], env=env, stdout=open('ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print(urllib.request.urlopen('http://localhost:11434/api/version').read().decode()); break
    except Exception:
        time.sleep(1)

In [ ]:
# 3. Pull the two models (same tags and quantisation as on the laptop)
!ollama pull qwen2.5:3b
!ollama pull qwen2.5:7b
!ollama list

In [ ]:
# 4. Cloudflare quick tunnel (no account): prints the public URL
import re
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:11434',
                           '--http-host-header', 'localhost:11434'],
                          stdout=open('tunnel.log', 'w'), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel.log').read())
    if m:
        url = m.group(0); break
    time.sleep(1)
print('P5_OLLAMA_URL =', url)

In [ ]:
# 5. Speed test through the tunnel (both models on the GPU?)
import json
for model in ['qwen2.5:3b', 'qwen2.5:7b']:
    body = {'model': model, 'stream': False, 'messages': [{'role': 'user', 'content': 'Say hi in five words.'}]}
    req = urllib.request.Request(url + '/api/chat', data=json.dumps(body).encode(), headers={'Content-Type': 'application/json'})
    t = time.time(); r = json.loads(urllib.request.urlopen(req, timeout=300).read())
    print(model, round(time.time() - t, 1), 's:', r['message']['content'],
          '| gen tok/s:', round(r['eval_count'] / (r['eval_duration'] / 1e9), 1))
!ollama ps

In [ ]:
# 6. Keep the session alive and show activity (stop this cell / the session when the runs are done)
while True:
    alive = server.poll() is None and tunnel.poll() is None
    last = open('ollama.log').read().strip().splitlines()[-1:]
    print(time.strftime('%H:%M:%S'), 'server+tunnel up' if alive else 'DOWN', '|', last[0][:120] if last else '', flush=True)
    time.sleep(60)